# 04 — From Predictions to Profits: Rule-Based Trading Strategies

## Week 5 Assignment

This notebook bridges the gap between ML-based price predictions and systematic trade execution.
You'll load the **five pre-trained models** from Week 4 and combine them with classic technical
rules to build four production-style strategies:

| # | Strategy | Models Used | Core Idea |
|---|---|---|---|
| 1 | **Dynamic Regime-Routed** | uptrend_lstm, downtrend_lstm, sideways_bilstm | Gating mechanism routes data to regime-specialist |
| 2 | **Momentum + LSTM Filter** | baseline_lstm | EMA trend + LSTM validation avoids bull traps |
| 3 | **Mean Reversion + BiLSTM** | sideways_bilstm | Bollinger / RSI oversold + BiLSTM bounce confirmation |
| 4 | **Volatility Breakout + ATR** | high_volatility_gru | GRU-predicted breakout sized by ATR, dynamic trailing stop |

A Buy-and-Hold benchmark is included for comparison.

**Your job:** the data loading, model architecture, and backtesting/evaluation engine are already
built for you. The four strategy functions and the master trend filter (Section 5 / 5b) are **not**
— look for `# TODO:` markers. Each one tells you *what* the function needs to decide; it's up to you
to work out *how*.

**A warning before you run anything:** the hyperparameters at the top of Section 0 are deliberately
bad starting points. Don't assume a negative Sharpe ratio or a wall of zeros means your logic is
wrong — it might just mean the knobs need turning. Implement first, tune second.

**Evaluation metrics:** Annualized Sharpe Ratio and Maximum Drawdown. You're aiming to beat
Buy-and-Hold on a risk-adjusted basis (Sharpe), not just on raw return.

**Key files consumed:**
- `data/processed/AAPL_features.csv`
- `models/baseline_lstm.pt`, `models/uptrend_lstm.pt`, `models/downtrend_lstm.pt`, `models/sideways_bilstm.pt`, `models/high_volatility_gru.pt`
- `models/baseline_scaler.pkl`, `models/regime_scaler.pkl`


---
## 0. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

# ============================================================================
# Paths  (all relative to the Week 4 project root)
# ============================================================================
DATA_PATH        = 'data/processed/AAPL_features.csv'
MODELS_DIR       = 'models'
REPORTS_DIR      = 'reports'

BASELINE_SCALER  = os.path.join(MODELS_DIR, 'baseline_scaler.pkl')
REGIME_SCALER    = os.path.join(MODELS_DIR, 'regime_scaler.pkl')

MODEL_PATHS = {
    'baseline':        os.path.join(MODELS_DIR, 'baseline_lstm.pt'),
    'uptrend':         os.path.join(MODELS_DIR, 'uptrend_lstm.pt'),
    'downtrend':       os.path.join(MODELS_DIR, 'downtrend_lstm.pt'),
    'sideways':        os.path.join(MODELS_DIR, 'sideways_bilstm.pt'),
    'high_volatility': os.path.join(MODELS_DIR, 'high_volatility_gru.pt'),
}

# ============================================================================
# Strategy hyper-parameters
# ----------------------------------------------------------------------------
# NOTE: several of these are set to deliberately poor values. Implementing
# the TODOs correctly is necessary but NOT sufficient to get a good result --
# you will also need to come back here and tune these once your logic works.
# ============================================================================
SEQUENCE_LENGTH  = 20          # same window used during training
TRAIN_SPLIT      = 0.80        # test set starts at 80% of the data

# --- Strategy 1 : Regime-Routed -------------------------------------------
ALPHA_THRESHOLD  = 0.01        # minimum |predicted return| to act
CONFIRM_DAYS     = 1           # raw signal must persist this many days before a flip is accepted
REGIME_COUNTERTREND_DAMP = True

# --- Master trend filter (applied to S1 & S4, see Section 5b) --------------
TREND_FILTER_WINDOW = 10       # long-horizon SMA used as a directional bias gate
TREND_FILTER_BAND   = 0.02     # price must be this far from the SMA for the gate to engage

# --- Strategy 2 : Momentum + LSTM ------------------------------------------
EMA_WINDOW       = 10          # EMA used for trend detection
MOM_THRESHOLD    = 0.005       # deadband on predicted return
TREND_DEADBAND   = 0.003       # +-band around EMA treated as "no strong trend"

# --- Strategy 3 : Mean Reversion + BiLSTM -----------------------------------
RSI_OVERSOLD     = 0.10        # RSI level treated as oversold
RSI_OVERBOUGHT   = 0.90        # RSI level treated as overbought
BB_OVERSOLD      = 0.05        # bb_pct_b level treated as oversold
BB_OVERBOUGHT    = 0.95        # bb_pct_b level treated as overbought
CONFIRM_EPS      = 0.0005      # dead-zone used to gate the BiLSTM's opinion
EXTREME_RSI_OVERSOLD   = 0.20  # a much deeper oversold level than RSI_OVERSOLD
EXTREME_RSI_OVERBOUGHT = 0.80  # a much deeper overbought level than RSI_OVERBOUGHT
EXTREME_BB_OVERSOLD    = 0.02
EXTREME_BB_OVERBOUGHT  = 0.98
MEANREV_LONG_REGIMES   = ('sideways', 'uptrend')    # regimes where a dip-buy is a reasonable prior
MEANREV_SHORT_REGIMES  = ('sideways', 'downtrend')  # regimes where fading a rip is a reasonable prior
MEANREV_MIN_HOLD       = 3     # bars to hold a reversion trade before a non-reversal exit is allowed
MEANREV_STOP_LOSS_PCT  = 0.04  # dedicated stop-loss for this strategy

# --- Strategy 4 : Volatility Breakout + ATR --------------------------------
ATR_MULTIPLIER_K = 3.0         # how many ATRs the predicted move must clear to count as a breakout
ATR_TRAIL_MULT   = 0.5         # trailing-stop distance, in ATRs
REENTRY_COOLDOWN = 2           # bars to wait before re-entering after a stop-out

# --- Global risk management (applied in the backtester) -------------------
HARD_STOP_LOSS_PCT = 0.06      # forced flat if a position is down >6% from its entry price
VOL_TARGET_ANNUAL  = 0.20      # target annualised volatility for the strategy's realised exposure
VOL_LOOKBACK       = 20        # trading days used to estimate realised volatility for sizing
MAX_LEVERAGE       = 1.0       # cap on |position size| after volatility scaling

TRANSACTION_COST = 0.0005      # 0.05% per trade (one-way)
RISK_FREE_RATE   = 0.02 / 252  # 2% annual -> daily

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'Data   : {DATA_PATH}')
print(f'Models : {list(MODEL_PATHS.keys())}')

# Dark plotting style  -- consistent with Weeks 1-4
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor':  '#0d1117',
    'axes.facecolor':    '#0d1117',
    'savefig.facecolor': '#0d1117',
    'axes.grid': True,
    'grid.alpha': 0.20,
    'font.family': 'monospace',
})


---
## 1. Data Loading Pipeline

Load the pre-processed feature CSV (same file used during training).  
The test partition is the last **20%** of rows — identical to the split used in Weeks 1-4.

In [ ]:
FEATURE_COLUMNS = [
    'return_1', 'return_5', 'log_return_1',
    'sma_ratio_20', 'sma_ratio_50', 'ema_ratio_20',
    'rsi_14', 'macd', 'macd_signal', 'macd_hist',
    'bb_width', 'bb_pct_b',
    'roc_5', 'roc_10',
    'atr_ratio',
    'stoch_k', 'stoch_d',
    'obv_change',
    'vwap_ratio',
    'z_score_20',
]

# --------------------------------------------------------------------------
# Load & sort
# --------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df = df.dropna().reset_index(drop=True)

print(f'Loaded {len(df)} rows | {df["date"].min().date()} -> {df["date"].max().date()}')
df[FEATURE_COLUMNS + ['close', 'atr_14', 'bb_pct_b', 'rsi_14', 'sma_ratio_50', 'macd', 'future_return']].head()

In [ ]:
# --------------------------------------------------------------------------
# Train / test split  (80 / 20  -- same as training)
# --------------------------------------------------------------------------
split_idx = int(len(df) * TRAIN_SPLIT)
df_train  = df.iloc[:split_idx].copy()
df_test   = df.iloc[split_idx:].copy().reset_index(drop=True)

print(f'Train rows : {len(df_train)}  ({df_train["date"].min().date()} -> {df_train["date"].max().date()})')
print(f'Test  rows : {len(df_test)}   ({df_test["date"].min().date()}  -> {df_test["date"].max().date()})')

---
## 2. Model Architecture (identical to Week 4)

Re-declaring the same `ReturnSequenceModel` class so the saved weights can be loaded
without importing any Week 4 notebook.

In [ ]:
class TemporalAttention(nn.Module):
    """Soft attention over the LSTM/GRU output sequence."""
    def __init__(self, hidden_size: int) -> None:
        super().__init__()
        self.scorer = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, rnn_out: torch.Tensor) -> torch.Tensor:
        scores  = self.scorer(rnn_out)          # (B, T, 1)
        weights = torch.softmax(scores, dim=1)  # normalise over time
        return (rnn_out * weights).sum(dim=1)   # (B, hidden) context vector


class ReturnSequenceModel(nn.Module):
    """
    Unified LSTM / GRU / BiLSTM backbone with temporal attention
    and a small regression head.  Matches the architecture saved in Week 4.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2,
                 dropout=0.4, model_type='lstm') -> None:
        super().__init__()
        model_type    = model_type.lower()
        bidirectional = model_type == 'bilstm'
        recurrent_cls = nn.GRU if model_type == 'gru' else nn.LSTM
        self.recurrent = recurrent_cls(
            input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True, bidirectional=bidirectional,
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.attention = TemporalAttention(out_dim)
        self.head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.recurrent(x)
        ctx    = self.attention(out)
        return self.head(ctx).squeeze(-1)


print('ReturnSequenceModel class defined OK.')

---
## 3. Load Scalers & Model Weights

In [ ]:
# --------------------------------------------------------------------------
# Scalers
# --------------------------------------------------------------------------
def load_scaler(path, fallback_df, cols):
    """Load saved scaler if it exists; otherwise fit one on the training split."""
    if os.path.exists(path):
        with open(path, 'rb') as fh:
            scaler = pickle.load(fh)
        print(f'  Loaded scaler from {path}')
    else:
        print(f'  {path} not found -- fitting scaler on train split.')
        scaler = StandardScaler().fit(fallback_df[cols])
    return scaler

print('Loading scalers...')
baseline_scaler = load_scaler(BASELINE_SCALER, df_train, FEATURE_COLUMNS)
regime_scaler   = load_scaler(REGIME_SCALER,   df_train, FEATURE_COLUMNS)

In [ ]:
# --------------------------------------------------------------------------
# Model hyper-parameters  (must match Week 4 RegimeConfig presets exactly)
# --------------------------------------------------------------------------
MODEL_CONFIGS = {
    'baseline':        dict(hidden_size=64, num_layers=2, dropout=0.40, model_type='lstm'),
    'uptrend':         dict(hidden_size=64, num_layers=2, dropout=0.40, model_type='lstm'),
    'downtrend':       dict(hidden_size=48, num_layers=2, dropout=0.45, model_type='lstm'),
    'sideways':        dict(hidden_size=64, num_layers=2, dropout=0.40, model_type='bilstm'),
    'high_volatility': dict(hidden_size=32, num_layers=1, dropout=0.50, model_type='gru'),
}

INPUT_SIZE = len(FEATURE_COLUMNS)  # 20 features

def load_model(name):
    cfg   = MODEL_CONFIGS[name]
    path  = MODEL_PATHS[name]
    model = ReturnSequenceModel(input_size=INPUT_SIZE, **cfg).to(DEVICE)
    if os.path.exists(path):
        state = torch.load(path, map_location=DEVICE)
        model.load_state_dict(state)
        print(f'  OK  {name:18s}  <- {path}')
    else:
        raise FileNotFoundError(f'Model weights not found: {path}')
    model.eval()
    return model

print('Loading model weights...')
models = {name: load_model(name) for name in MODEL_CONFIGS}
print(f'\nAll {len(models)} models loaded successfully.')

---
## 4. Signal Generation Engine

Core helpers: build sliding-window sequences, run batched inference, classify regimes.

In [ ]:
# --------------------------------------------------------------------------
# Helper: sliding-window sequence builder
# --------------------------------------------------------------------------
def make_sequences(scaled_features, seq_len):
    """
    Slide a window of `seq_len` rows.
    Returns shape (N - seq_len, seq_len, n_features).
    Sequence i predicts for original row (i + seq_len).
    """
    xs = []
    for i in range(seq_len, len(scaled_features)):
        xs.append(scaled_features[i - seq_len : i])
    return np.asarray(xs, dtype=np.float32)


@torch.no_grad()
def predict_returns(model, sequences, batch_size=512):
    """
    Mini-batch inference; returns 1-D numpy array of predicted next-day returns.
    """
    preds = []
    for i in range(0, len(sequences), batch_size):
        batch = torch.tensor(sequences[i : i + batch_size]).to(DEVICE)
        preds.append(model(batch).cpu().numpy())
    return np.concatenate(preds)


# --------------------------------------------------------------------------
# Regime classifier  (identical to Week 4 notebook 2)
# --------------------------------------------------------------------------
def classify_regimes(data, vol_quantile=0.75, smooth_window=5):
    """
    Returns a string array: 'uptrend', 'downtrend', 'high_volatility', 'sideways'.
    """
    realized_vol  = data['return_1'].rolling(10).std().bfill()
    vol_threshold = realized_vol.quantile(vol_quantile)
    high_vol_flag = realized_vol >= vol_threshold

    trend_threshold = data['sma_ratio_50'].std() * 0.5
    trend = pd.Series('sideways', index=data.index)
    trend[data['sma_ratio_50'] >  trend_threshold] = 'uptrend'
    trend[data['sma_ratio_50'] < -trend_threshold] = 'downtrend'
    trend[(trend == 'uptrend')   & (data['macd'] < 0)] = 'sideways'
    trend[(trend == 'downtrend') & (data['macd'] > 0)] = 'sideways'

    regime = trend.copy()
    regime[(trend == 'sideways') & high_vol_flag] = 'high_volatility'

    # 5-day rolling majority vote
    codes, uniques = pd.factorize(regime)
    smoothed = (
        pd.Series(codes)
        .rolling(smooth_window, center=True, min_periods=1)
        .apply(lambda x: np.bincount(x.astype(int), minlength=len(uniques)).argmax())
        .astype(int).values
    )
    return uniques[smoothed]


print('Signal-generation helpers defined OK.')

In [ ]:
# --------------------------------------------------------------------------
# Pre-compute scaled features and sequences for the FULL dataset
# so each strategy can draw from cached arrays (no repeated transforms).
# --------------------------------------------------------------------------
print('Pre-computing scaled features and sequences...')

# Baseline scaler  (used by strategies 2 & 3 via baseline_lstm)
scaled_all_baseline = baseline_scaler.transform(df[FEATURE_COLUMNS])
seqs_baseline       = make_sequences(scaled_all_baseline, SEQUENCE_LENGTH)

# Regime scaler  (used by strategies 1 & 4)
scaled_all_regime = regime_scaler.transform(df[FEATURE_COLUMNS])
seqs_regime       = make_sequences(scaled_all_regime, SEQUENCE_LENGTH)

# Sequence index j maps to df row (j + SEQUENCE_LENGTH).
# For df_test row i  (absolute df index = split_idx + i):
#   sequence index = (split_idx + i) - SEQUENCE_LENGTH
N_test = len(df_test)
test_seq_indices = np.arange(
    split_idx - SEQUENCE_LENGTH,
    split_idx - SEQUENCE_LENGTH + N_test
)
# Guard against overflow
max_valid        = min(len(seqs_baseline), len(seqs_regime))
test_seq_indices = test_seq_indices[test_seq_indices < max_valid]
N_test_eff       = len(test_seq_indices)

# Align df_test to the effective window
df_test_eff = df_test.iloc[:N_test_eff].copy().reset_index(drop=True)

print(f'Total sequences (baseline) : {len(seqs_baseline)}')
print(f'Total sequences (regime)   : {len(seqs_regime)}')
print(f'Effective test days        : {N_test_eff}  '
      f'({df_test_eff["date"].min().date()} -> {df_test_eff["date"].max().date()})')

In [ ]:
# --------------------------------------------------------------------------
# Run inference for all models on the test-set sequences
# --------------------------------------------------------------------------
print('Running inference on test sequences...')

test_seqs_baseline = seqs_baseline[test_seq_indices]   # (N_test_eff, 20, 20)
test_seqs_regime   = seqs_regime[test_seq_indices]

preds = {}
for name, mdl in models.items():
    seqs = test_seqs_baseline if name == 'baseline' else test_seqs_regime
    preds[name] = predict_returns(mdl, seqs)
    print(f'  {name:18s}  shape={preds[name].shape}  '
          f'mean={preds[name].mean():.5f}  std={preds[name].std():.5f}')

# Classify regime for every test day
regimes_test = classify_regimes(df_test_eff)
print('\nRegime distribution on test set:')
for r in ['uptrend', 'downtrend', 'high_volatility', 'sideways']:
    n = int((regimes_test == r).sum())
    print(f'  {r:18s}: {n:4d} days ({100*n/N_test_eff:.1f}%)')

---
## 5. Four Rule-Based Strategies

Each strategy produces a **signals** array of `{-1, 0, +1}` (short / flat / long) over the test set.
Complete the `# TODO:` sections in each function below — the docstring-style comments above each
`for` loop tell you what decision the code needs to make, not how to make it.


### Strategy 1: Dynamic Regime-Routed

Each day, the classified regime picks which specialist model's prediction to trust. Your job is to
turn that model's predicted return into a disciplined trading signal — one that doesn't flip
direction every time the prediction wobbles across zero.

Think about two separate problems here: (1) turning a single day's prediction into a directional
call, and (2) deciding whether a single day's call is enough to actually change your position.

> Hint: a raw signal and a final signal don't have to be the same array.

> Hint: "fighting the regime" doesn't have to mean "banned outright" — how confident does the
> model need to be before you'll let it override the regime?


In [ ]:
# ============================================================================
# Strategy 1  --  Dynamic Regime-Routed
# ============================================================================

def strategy_regime_routed(preds_dict, regimes, alpha=ALPHA_THRESHOLD,
                            confirm_days=CONFIRM_DAYS,
                            damp_countertrend=REGIME_COUNTERTREND_DAMP):
    REGIME_MODEL = {
        'uptrend':         'uptrend',
        'downtrend':       'downtrend',
        'sideways':        'sideways',
        'high_volatility': 'high_volatility',
    }
    N           = len(regimes)
    raw_signal  = np.zeros(N, dtype=float)
    pred_return = np.zeros(N, dtype=float)

    for i, regime in enumerate(regimes):
        model_key      = REGIME_MODEL.get(regime, 'baseline')
        p              = float(preds_dict[model_key][i])
        pred_return[i] = p

        # TODO 1a: Convert `p` into a raw directional call: +1 / -1 / 0.
        #   What role should `alpha` play?

        # TODO 1b (optional but recommended): damp -- don't necessarily ban --
        #   a raw call that fights the current `regime`, using
        #   `damp_countertrend` to control whether this is active.

    # TODO 1c: Raw signals can flip every single day. Build `signals` so that
    #   a NEW direction is only accepted once the raw signal has agreed with
    #   itself for `confirm_days` consecutive bars. (Should dropping to flat
    #   need the same confirmation as taking a new direction?)
    signals = np.zeros(N, dtype=float)

    return signals, pred_return


sig1, pred1 = strategy_regime_routed(preds, regimes_test)
print('Strategy 1 -- Regime Routed')
print(f'  Long: {(sig1==1).sum()}  Short: {(sig1==-1).sum()}  Flat: {(sig1==0).sum()}')


### Strategy 2: Momentum + LSTM Filter

Go long when the LSTM predicts an up move, go short when it predicts a down move — but only if
you're not fighting a *real* trend in the wrong direction. The tricky part is defining "real."

> Hint: two deadbands are sitting in the function signature. What is each one protecting against —
> a noisy prediction, or a noisy trend reading?

> Hint: should the trend requirement be something the LSTM prediction must *satisfy*, or something
> it only needs to *not clearly violate*? Those produce very different trade counts.


In [ ]:
# ============================================================================
# Strategy 2  --  Momentum Filtered by LSTM Prediction
# ============================================================================

def strategy_momentum_lstm(df_full, df_test_slice, pred_baseline,
                            ema_window=EMA_WINDOW,
                            mom_threshold=MOM_THRESHOLD,
                            trend_deadband=TREND_DEADBAND):
    ema50_full = df_full['close'].ewm(span=ema_window, adjust=False).mean()
    ema50_test = ema50_full.iloc[split_idx : split_idx + len(df_test_slice)].values
    close_test = df_test_slice['close'].values

    N = len(df_test_slice)
    signals = np.zeros(N, dtype=float)

    for i in range(N):
        trend_score = (close_test[i] / ema50_test[i]) - 1.0   # signed % distance from EMA
        pred        = pred_baseline[i]

        # TODO 2: Combine `trend_score` and `pred` into a signal (+1 / -1 / 0).
        #   The strategy should generally follow the LSTM's prediction, but
        #   should not take a trade that directly fights an established trend.
        #   Decide what "established" means for both the trend and the
        #   prediction, and whether the trend should gate the signal or the
        #   other way around.
        pass

    return signals


sig2 = strategy_momentum_lstm(df, df_test_eff, preds['baseline'])
print('Strategy 2 -- Momentum + LSTM Filter')
print(f'  Long: {(sig2==1).sum()}  Short: {(sig2==-1).sum()}  Flat: {(sig2==0).sum()}')


### Strategy 3: Mean Reversion + BiLSTM

Buy oversold, short overbought, use the BiLSTM to keep you out of bad reversals. Simple in
principle — but a plain oversold reading during a real crash isn't a bounce setup, it's a falling
knife, and this dataset's test window has one. You'll need something beyond RSI/BB thresholds
alone to tell the difference.

You have four ingredients available: two "conviction levels" (normal and extreme) for how
oversold/overbought is oversold/overbought enough, the classified market regime, and the BiLSTM's
opinion. Think about how conviction level, regime, and model confirmation should interact — does
a normal-conviction setup need the same bar of proof as an extreme one?

> Hint: "the model didn't disagree" and "the model actively agrees" are two different bars. When
> might you want to demand the stronger one?

> Hint: once a reversion trade is open, should one noisy day immediately flatten it? Consider
> giving a trade a minimum number of bars to work before anything but a genuine reversal can exit it.


In [ ]:
# ============================================================================
# Strategy 3  --  Mean Reversion with BiLSTM Validation
# ============================================================================

def strategy_mean_reversion_bilstm(df_test_slice, pred_sideways, regimes,
                                    rsi_oversold=RSI_OVERSOLD,
                                    rsi_overbought=RSI_OVERBOUGHT,
                                    bb_oversold=BB_OVERSOLD,
                                    bb_overbought=BB_OVERBOUGHT,
                                    confirm_eps=CONFIRM_EPS,
                                    extreme_rsi_oversold=EXTREME_RSI_OVERSOLD,
                                    extreme_rsi_overbought=EXTREME_RSI_OVERBOUGHT,
                                    extreme_bb_oversold=EXTREME_BB_OVERSOLD,
                                    extreme_bb_overbought=EXTREME_BB_OVERBOUGHT,
                                    long_regimes=MEANREV_LONG_REGIMES,
                                    short_regimes=MEANREV_SHORT_REGIMES,
                                    min_hold=MEANREV_MIN_HOLD):
    bb_pct_b = df_test_slice['bb_pct_b'].values
    rsi      = df_test_slice['rsi_14'].values
    N = len(df_test_slice)
    raw = np.zeros(N, dtype=float)

    for i in range(N):
        # TODO 3a: Define normal-conviction and extreme-conviction
        #   oversold/overbought conditions using `rsi`, `bb_pct_b`, and the
        #   two sets of thresholds passed into this function.

        # TODO 3b: Decide how much the BiLSTM (`pred_sideways[i]`) needs to
        #   agree with a trade before it's allowed, using `confirm_eps`.
        #   Consider whether this bar should be the same for every setup.

        # TODO 3c: Gate a normal-conviction trade by whether the current
        #   regime (`regimes[i]`, `long_regimes`, `short_regimes`) is one
        #   where reversion is a reasonable prior. Then decide whether an
        #   extreme-conviction reading should ever be allowed to act even
        #   when the regime gate says no.
        pass

    # TODO 3d: Turn `raw` into `signals` with a minimum holding period
    #   (`min_hold`): once a position opens, only an actual reversal in
    #   `raw` should be able to exit it early.
    signals = np.zeros(N, dtype=float)

    return signals


sig3 = strategy_mean_reversion_bilstm(df_test_eff, preds['sideways'], regimes_test)
print('Strategy 3 -- Mean Reversion + BiLSTM')
print(f'  Long: {(sig3==1).sum()}  Short: {(sig3==-1).sum()}  Flat: {(sig3==0).sum()}')


### Strategy 4: Volatility Breakout + ATR

Enter when the GRU's predicted price clears an ATR-scaled band; manage the trade with a trailing
stop plus a hard percentage backstop; get out of the way for a cooldown period after a stop-out.

Pay close attention to *order of operations* within a single day's loop iteration. A stop-out and
a fresh entry are two different decisions — think about whether they should ever be allowed to
happen on the same bar, and why.

> Hint: a trailing stop should only ever move in the direction that protects more of your gains.
> `max()` / `min()` (depending on the direction you're in) enforce that in one line.

> Hint: if you reset `position` to 0 the moment a stop triggers, what stops the entry check right
> below it from firing again immediately?


In [ ]:
# ============================================================================
# Strategy 4  --  Volatility Breakout with ATR Bands & Trailing Stop
# ============================================================================

def strategy_volatility_breakout_atr(df_test_slice, pred_gru,
                                     k=ATR_MULTIPLIER_K,
                                     trail_mult=ATR_TRAIL_MULT,
                                     cooldown_bars=REENTRY_COOLDOWN,
                                     hard_stop_pct=HARD_STOP_LOSS_PCT):
    close = df_test_slice['close'].values
    atr   = df_test_slice['atr_14'].values
    N     = len(df_test_slice)

    signals        = np.zeros(N, dtype=float)
    position       = 0
    trailing_stop  = np.nan
    entry_price    = np.nan
    cooldown_timer = 0

    for i in range(N):
        pt         = close[i]
        atr_t      = atr[i]
        pred_price = pt * (1.0 + float(pred_gru[i]))

        upper_band = pt + k * atr_t
        lower_band = pt - k * atr_t

        stopped_out_this_bar = False

        # TODO 4a: If a position is open, manage its exit:
        #   - a trailing stop (`trailing_stop`) that only ever tightens as
        #     price moves in your favor, using `trail_mult * atr_t`
        #   - a hard percentage stop-loss (`hard_stop_pct`) off `entry_price`
        #     as a backstop
        #   Reset `position`, `trailing_stop`, `entry_price` and set
        #   `stopped_out_this_bar = True` if either one triggers.
        if position == 1:
            pass
        elif position == -1:
            pass

        if stopped_out_this_bar:
            cooldown_timer = cooldown_bars

        # TODO 4b: Entry logic -- only when flat AND not in cooldown. Compare
        #   `pred_price` to `upper_band` / `lower_band` to decide long/short,
        #   and initialize `trailing_stop` / `entry_price` on entry.
        if position == 0 and cooldown_timer == 0:
            pass
        elif cooldown_timer > 0:
            cooldown_timer -= 1

        signals[i] = position

    return signals


sig4 = strategy_volatility_breakout_atr(df_test_eff, preds['high_volatility'])
print('Strategy 4 -- Volatility Breakout + ATR Trailing Stop')
print(f'  Long: {(sig4==1).sum()}  Short: {(sig4==-1).sum()}  Flat: {(sig4==0).sum()}')


---
## 5b. Master Trend Filter

Strategies 1 and 4 can both go net short. Think about what a short position is worth during a
strong, sustained rally — and what a long position is worth during a strong, sustained decline.
Build a filter that vetoes a trade fighting an *established* trend, while leaving everything else
(including trades inside a normal, range-bound stretch) untouched.

> Hint: "established" should probably mean more than "one tick above the average."

> Hint: what should happen to a signal that's already flat?


In [ ]:
# ============================================================================
# 5b. Master Trend Filter  --  applied to Strategy 1 & Strategy 4
# ============================================================================

def apply_master_trend_filter(signal, close_prices, window=TREND_FILTER_WINDOW,
                               band=TREND_FILTER_BAND):
    close  = pd.Series(close_prices)
    sma    = close.rolling(window, min_periods=1).mean().values
    price  = close.values

    # TODO 5: Using `price`, `sma`, and `band`, identify bars that are in an
    #   established uptrend and bars that are in an established downtrend.
    #   Veto shorts (`signal == -1`) in the former and longs (`signal == 1`)
    #   in the latter; leave everything else in `signal` unchanged.
    filtered = signal.copy()

    return filtered


sig1_raw, sig4_raw = sig1.copy(), sig4.copy()
sig1 = apply_master_trend_filter(sig1, close_prices=df_test_eff['close'].values)
sig4 = apply_master_trend_filter(sig4, close_prices=df_test_eff['close'].values)

for name, raw, filt in [('Strategy 1', sig1_raw, sig1), ('Strategy 4', sig4_raw, sig4)]:
    vetoed_shorts = int(((raw == -1) & (filt == 0)).sum())
    vetoed_longs  = int(((raw ==  1) & (filt == 0)).sum())
    print(f'{name}: vetoed {vetoed_shorts} counter-trend shorts, {vetoed_longs} counter-trend longs')
    print(f'  New counts -> Long: {(filt==1).sum()}  Short: {(filt==-1).sum()}  Flat: {(filt==0).sum()}')


---
## 6. Vectorized Backtesting with Transaction Costs

Each strategy is simulated on the test set:
1. Daily market returns are derived from closing prices.
2. They are multiplied by the **lagged** signal (signal set at close of day $t-1$, realised at close of day $t$).
3. A **transaction cost drag** of 0.05% per trade (one-way) is subtracted whenever the position changes.

The engine itself is built for you -- two small formula-driven `# TODO:` gaps are marked inline
(volatility-targeted sizing, and the transaction-cost drag). Everything else (the stop-loss loop,
entry-price tracking) is given as working infrastructure.

In [ ]:
# ============================================================================
# Risk-managed backtester  (REPLACES the plain vectorized_backtest)
# ============================================================================
# This engine is given to you fully working, with two exceptions marked
# below -- both are "plug in the formula" TODOs, not open-ended design
# decisions like Section 5.
# ============================================================================

def risk_managed_backtest(signals, close_prices, tc=TRANSACTION_COST,
                           hard_stop_pct=HARD_STOP_LOSS_PCT,
                           vol_target=VOL_TARGET_ANNUAL,
                           vol_lookback=VOL_LOOKBACK,
                           max_leverage=MAX_LEVERAGE):
    n          = len(signals)
    market_ret = np.diff(close_prices, prepend=close_prices[0]) / close_prices
    market_ret[0] = 0.0

    # Realized vol of the underlying, annualised, for position sizing.
    ret_series      = pd.Series(market_ret)
    realized_vol    = (ret_series.rolling(vol_lookback).std() * np.sqrt(252)).bfill().values
    realized_vol    = np.where(realized_vol < 1e-6, 1e-6, realized_vol)  # avoid div-by-zero

    # TODO (backtest-a): volatility-targeted position scalar.
    #   formula:  vol_scalar = clip(vol_target / realized_vol, 0, max_leverage)
    #   i.e. size UP in calm markets, size DOWN in violent ones, capped so you
    #   never exceed max_leverage.
    vol_scalar = np.ones(n)

    position       = np.zeros(n)
    entry_price    = np.nan
    open_direction = 0

    for i in range(1, n):
        raw_sig = signals[i - 1]           # decided at close of t-1
        px      = close_prices[i - 1]

        # Did we just get stopped out of an existing bar-(i-1) position?
        if open_direction == 1 and not np.isnan(entry_price):
            if px < entry_price * (1 - hard_stop_pct):
                raw_sig = 0
        elif open_direction == -1 and not np.isnan(entry_price):
            if px > entry_price * (1 + hard_stop_pct):
                raw_sig = 0

        # Track/refresh the entry price whenever direction changes
        if raw_sig != open_direction:
            open_direction = raw_sig
            entry_price    = px if raw_sig != 0 else np.nan

        position[i] = raw_sig * vol_scalar[i - 1]

    gross_ret = position * market_ret

    # TODO (backtest-b): transaction-cost drag.
    #   formula:  tc_drag[t] = |position[t] - position[t-1]| * tc
    #   (position[-1] is treated as 0 for the very first bar)
    tc_drag = np.zeros(n)

    return pd.Series(gross_ret - tc_drag, name='daily_return')


close_test = df_test_eff['close'].values

ret1 = risk_managed_backtest(sig1, close_test)
ret2 = risk_managed_backtest(sig2, close_test)
ret3 = risk_managed_backtest(sig3, close_test, hard_stop_pct=MEANREV_STOP_LOSS_PCT)  # tighter leash -- falling-knife risk
ret4 = risk_managed_backtest(sig4, close_test)

# Buy-and-hold benchmark (always long, unscaled -- for reference only)
market_ret_bh = pd.Series(
    np.diff(close_test, prepend=close_test[0]) / close_test,
    name='daily_return'
)
market_ret_bh.iloc[0] = 0.0

print('Backtesting complete (with vol-targeted sizing + hard stop-loss).')
print('Cumulative returns:')
for name, r in [('Strategy 1 (Regime Routed)', ret1),
                ('Strategy 2 (Momentum+LSTM)', ret2),
                ('Strategy 3 (Mean Reversion)', ret3),
                ('Strategy 4 (Volatility ATR)', ret4),
                ('Buy & Hold',                  market_ret_bh)]:
    cum = (1 + r).cumprod().iloc[-1] - 1
    print(f'  {name:<30s}  {cum*100:+.2f}%')


---
## 7. Performance Evaluation

Compute **Annualised Sharpe Ratio** and **Maximum Drawdown** for every strategy.

$$
\text{Sharpe Ratio} = \frac{E[R_p - R_f]}{\sigma(R_p - R_f)} \times \sqrt{252}
$$

$$
\text{Max Drawdown} = \max_t \left(1 - \frac{V_t}{\max_{s \le t} V_s}\right)
$$

where $R_f$ = 2% p.a. daily risk-free rate and $\sqrt{252}$ annualises the daily Sharpe.

Every function below has its formula spelled out in its docstring — implement the `# TODO:` body
to match it exactly (unit mismatches here will silently break every chart downstream).

In [ ]:
# ============================================================================
# Performance metrics -- formulas given, implementation is your job.
# ============================================================================

def sharpe_ratio(daily_returns, rf_daily=RISK_FREE_RATE):
    """
    Annualised Sharpe Ratio.
    formula: mean(daily_returns - rf_daily) / std(daily_returns - rf_daily) * sqrt(252)
    (return 0.0 if the standard deviation is 0, to avoid dividing by zero)
    """
    # TODO: implement using the formula above
    return 0.0


def max_drawdown(daily_returns):
    """
    Maximum Drawdown.
    formula:
        cum         = cumulative product of (1 + daily_returns)
        running_max = running maximum of cum
        drawdown    = (cum - running_max) / running_max
        MDD         = -min(drawdown)   (reported as a positive number)
    """
    # TODO: implement using the formula above
    return 0.0


def total_return(daily_returns):
    """
    formula: cumulative product of (1 + daily_returns), minus 1, at the final bar.
    """
    # TODO: implement using the formula above
    return 0.0


def annualised_return(daily_returns, trading_days=252):
    """
    Annualised (CAGR-style) return.
    formula: cum_final ** (trading_days / n) - 1
    where cum_final is the cumulative product of (1 + daily_returns) at the
    final bar, and n is the number of return observations.
    """
    # TODO: implement using the formula above
    return 0.0


def win_rate(signals, daily_mkt_ret):
    """
    Fraction of active trading days with positive P&L.
    formula:
        active = days where signal != 0
        pnl    = signal * daily_mkt_ret, restricted to active days
        win_rate = fraction of `pnl` values that are > 0
    (return 0.0 if there are no active days)
    """
    # TODO: implement using the formula above
    return 0.0


strategy_labels  = [
    'S1: Regime Routed',
    'S2: Momentum+LSTM',
    'S3: Mean Reversion',
    'S4: Volatility Breakout',
    'Buy & Hold (benchmark)',
]
strategy_returns = [ret1, ret2, ret3, ret4, market_ret_bh]
strategy_signals = [sig1, sig2, sig3, sig4, np.ones(N_test_eff)]
mkt_daily = np.diff(close_test, prepend=close_test[0]) / close_test

rows = []
for label, r, sig in zip(strategy_labels, strategy_returns, strategy_signals):
    rows.append({
        'Strategy':          label,
        'Total Return (%)':  round(total_return(r) * 100, 2),
        'Ann. Return (%)':   round(annualised_return(r) * 100, 2),
        'Sharpe Ratio':      round(sharpe_ratio(r), 3),
        'Max Drawdown (%)':  round(max_drawdown(r) * 100, 2),
        'Daily Std (%)':     round(r.std() * 100, 3),
        'Win Rate (%)':      round(win_rate(sig, mkt_daily) * 100, 1),
    })

perf_df = pd.DataFrame(rows).set_index('Strategy')

print('\n' + '=' * 80)
print('PERFORMANCE SUMMARY')
print('=' * 80)
print(perf_df.to_string())
print('=' * 80)


In [ ]:
# Colour-coded Pandas styled table
def colour_sharpe(val):
    c = '#2ecc71' if val > 0.5 else ('#e74c3c' if val < 0 else '#f39c12')
    return f'color: {c}; font-weight: bold'

def colour_mdd(val):
    c = '#2ecc71' if val < 10 else ('#e74c3c' if val > 25 else '#f39c12')
    return f'color: {c}'

(
    perf_df.style
    .applymap(colour_sharpe, subset=['Sharpe Ratio'])
    .applymap(colour_mdd,    subset=['Max Drawdown (%)'])
    .format({
        'Total Return (%)':  '{:+.2f}%',
        'Ann. Return (%)':   '{:+.2f}%',
        'Sharpe Ratio':      '{:.3f}',
        'Max Drawdown (%)':  '{:.2f}%',
        'Daily Std (%)':     '{:.3f}%',
        'Win Rate (%)':      '{:.1f}%',
    })
    .set_caption('Strategy Performance Comparison -- Test Set')
)

---
## 8. Visualisation

Five figures are generated and saved to `reports/`:
1. **Cumulative Returns** — headline chart
2. **Drawdown Profiles** — risk trajectory
3. **Signal Maps** — entry/exit overlaid on price
4. **Sharpe & MDD Bar Charts** — at-a-glance ranking
5. **Monthly Return Heatmap** — seasonality analysis

Figures 1–4 each have a small `# TODO:` where a quantity needs to be derived from `strategy_returns`
/ `perf_df` before it can be plotted — the formula is given each time, plotting code is not the
exercise. Figure 5 is provided complete.

In [ ]:
# ============================================================
# Figure 1 -- Cumulative Returns
# ============================================================
COLORS = {
    'S1: Regime Routed':          '#a78bfa',
    'S2: Momentum+LSTM':          '#34d399',
    'S3: Mean Reversion':         '#fb923c',
    'S4: Volatility Breakout':    '#f472b6',
    'Buy & Hold (benchmark)':     '#94a3b8',
}

dates = pd.to_datetime(df_test_eff['date'].values)

fig, ax = plt.subplots(figsize=(15, 7))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

for label, r in zip(strategy_labels, strategy_returns):
    # TODO: compute the cumulative return series `cum` for this strategy.
    #   formula: cumulative product of (1 + r), minus 1
    cum = np.zeros(len(r))

    lw  = 1.5 if 'benchmark' in label else 2.5
    ls  = '--' if 'benchmark' in label else '-'
    ax.plot(dates, cum * 100, label=label, color=COLORS[label], lw=lw, ls=ls, alpha=0.92)

ax.axhline(0, color='#475569', lw=0.8, ls=':')
ax.set_title('Cumulative Returns -- Rule-Based Strategies vs Buy & Hold',
             color='white', fontsize=15, pad=14)
ax.set_xlabel('Date', color='#94a3b8')
ax.set_ylabel('Cumulative Return (%)', color='#94a3b8')
ax.tick_params(colors='#94a3b8')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:+.0f}%'))
ax.legend(loc='upper left', framealpha=0.25, fontsize=9)
ax.grid(True, alpha=0.15)
for spine in ax.spines.values():
    spine.set_edgecolor('#334155')

plt.tight_layout()
fig.savefig(os.path.join(REPORTS_DIR, 'fig_cumulative_returns.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/fig_cumulative_returns.png')


In [ ]:
# ============================================================
# Figure 2 -- Drawdown Profiles
# ============================================================
fig, axes = plt.subplots(len(strategy_labels), 1,
                          figsize=(15, 3.0 * len(strategy_labels)),
                          sharex=True)
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Drawdown Profiles per Strategy', color='white', fontsize=14, y=1.002)

for ax, label, r in zip(axes, strategy_labels, strategy_returns):
    ax.set_facecolor('#0d1117')

    # TODO: compute the running drawdown series `drawdown` (in %) for this strategy.
    #   formula:
    #     cum      = cumulative product of (1 + r)
    #     peak     = running maximum of cum
    #     drawdown = (cum - peak) / peak * 100
    drawdown = pd.Series(np.zeros(len(r)))

    ax.fill_between(dates, drawdown, 0, color=COLORS[label], alpha=0.35)
    ax.plot(dates, drawdown, color=COLORS[label], lw=1.0)
    ax.set_ylabel(f'{label}\nDD (%)', color='#94a3b8', fontsize=8)
    ax.axhline(0, color='#475569', lw=0.6, ls=':')
    ax.tick_params(colors='#94a3b8', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#334155')

axes[-1].set_xlabel('Date', color='#94a3b8')
plt.tight_layout()
fig.savefig(os.path.join(REPORTS_DIR, 'fig_drawdown_profiles.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/fig_drawdown_profiles.png')


In [ ]:
# ============================================================
# Figure 3 -- Signal Maps
# ============================================================
all_signals  = [sig1, sig2, sig3, sig4]
signal_names = strategy_labels[:4]

fig, axes = plt.subplots(4, 1, figsize=(15, 16), sharex=True)
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Signal Maps: Price + Entry Signals', color='white', fontsize=14, y=1.002)

for ax, sig, name in zip(axes, all_signals, signal_names):
    ax.set_facecolor('#0d1117')
    ax.plot(dates, close_test, color='#e2e8f0', lw=0.8, label='Close')

    # TODO: find the day-indices where this strategy is Long (`sig == 1`) and
    #   Short (`sig == -1`). `np.where(...)` returns the indices you need.
    long_idx  = np.array([], dtype=int)
    short_idx = np.array([], dtype=int)

    if len(long_idx):
        ax.scatter(dates[long_idx],  close_test[long_idx],
                   color='#2ecc71', s=12, zorder=3, label='Long', alpha=0.75, marker='^')
    if len(short_idx):
        ax.scatter(dates[short_idx], close_test[short_idx],
                   color='#e74c3c', s=12, zorder=3, label='Short', alpha=0.75, marker='v')

    ax.set_ylabel('Price', color='#94a3b8', fontsize=9)
    ax.set_title(name, color='#e2e8f0', fontsize=10, pad=4)
    ax.tick_params(colors='#94a3b8', labelsize=8)
    ax.legend(fontsize=8, framealpha=0.2)
    ax.grid(True, alpha=0.12)
    for spine in ax.spines.values():
        spine.set_edgecolor('#334155')

axes[-1].set_xlabel('Date', color='#94a3b8')
plt.tight_layout()
fig.savefig(os.path.join(REPORTS_DIR, 'fig_signal_maps.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/fig_signal_maps.png')


In [ ]:
# ============================================================
# Figure 4 -- Sharpe & MDD Bar Charts
# ============================================================
short_labels = ['S1','S2','S3','S4','B&H']

# TODO: pull the two columns you need straight out of `perf_df`
#   (Sharpe Ratio -> sharpe_vals, Max Drawdown (%) -> mdd_vals).
sharpe_vals = np.zeros(len(short_labels))
mdd_vals    = np.zeros(len(short_labels))

bar_colors   = [COLORS[l] for l in strategy_labels]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

for ax in (ax1, ax2):
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='#94a3b8')
    ax.grid(True, alpha=0.12, axis='y')
    for spine in ax.spines.values():
        spine.set_edgecolor('#334155')

bars1 = ax1.bar(short_labels, sharpe_vals, color=bar_colors,
                edgecolor='#334155', linewidth=0.8)
ax1.axhline(0, color='#475569', lw=0.8, ls=':')
ax1.set_title('Annualised Sharpe Ratio', color='white', fontsize=12)
ax1.set_ylabel('Sharpe', color='#94a3b8')
for bar, val in zip(bars1, sharpe_vals):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.02 * np.sign(val) + 0.01,
             f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=9)

bars2 = ax2.bar(short_labels, mdd_vals, color=bar_colors,
                edgecolor='#334155', linewidth=0.8)
ax2.set_title('Maximum Drawdown (%)', color='white', fontsize=12)
ax2.set_ylabel('Max DD (%)', color='#94a3b8')
for bar, val in zip(bars2, mdd_vals):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', color='white', fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(REPORTS_DIR, 'fig_sharpe_mdd.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/fig_sharpe_mdd.png')


In [ ]:
# ============================================================
# Figure 5 -- Monthly Return Heatmap
# ============================================================

def monthly_returns_pivot(daily_ret, date_idx):
    s = daily_ret.copy()
    s.index = pd.DatetimeIndex(date_idx)
    monthly = s.resample('ME').apply(lambda x: (1 + x).prod() - 1)
    dm = monthly.to_frame(name='ret')
    dm['year']  = dm.index.year
    dm['month'] = dm.index.month
    return dm.pivot(index='year', columns='month', values='ret') * 100


month_abbr = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(len(strategy_labels), 1,
                          figsize=(14, 3.0 * len(strategy_labels)))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Monthly Returns Heatmap (%)', color='white', fontsize=14, y=1.002)

for ax, label, r in zip(axes, strategy_labels, strategy_returns):
    ax.set_facecolor('#0d1117')
    mr  = monthly_returns_pivot(r, dates)
    arr = mr.values
    vmax = max(np.nanpercentile(np.abs(arr), 95), 1e-6)

    im = ax.imshow(arr, cmap='RdYlGn', aspect='auto',
                   vmin=-vmax, vmax=vmax, interpolation='nearest')
    ax.set_yticks(range(len(mr.index)))
    ax.set_yticklabels(mr.index.tolist(), color='#94a3b8', fontsize=8)
    ax.set_xticks(range(len(mr.columns)))
    ax.set_xticklabels([month_abbr[c-1] for c in mr.columns],
                       color='#94a3b8', fontsize=8)
    ax.set_title(label, color='#e2e8f0', fontsize=9, pad=4)

    for y_i in range(arr.shape[0]):
        for x_i in range(arr.shape[1]):
            val = arr[y_i, x_i]
            if not np.isnan(val):
                txt_color = 'white' if abs(val) > vmax * 0.5 else 'black'
                ax.text(x_i, y_i, f'{val:.1f}', ha='center', va='center',
                        color=txt_color, fontsize=6.5)

    fig.colorbar(im, ax=ax, shrink=0.8, format='%.1f%%')

plt.tight_layout()
fig.savefig(os.path.join(REPORTS_DIR, 'fig_monthly_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/fig_monthly_heatmap.png')

---
## 9. Final Performance Table & Summary

In [ ]:
# Save performance table to CSV
out_csv = os.path.join(REPORTS_DIR, 'week5_strategy_performance.csv')
perf_df.to_csv(out_csv)
print(f'Saved: {out_csv}')

print('\n' + '=' * 80)
print('WEEK 5 FINAL PERFORMANCE COMPARISON')
print('=' * 80)
print(perf_df.to_string())
print('=' * 80)
print()

strat_only = perf_df.iloc[:-1]   # exclude Buy & Hold from "best strategy" ranking

# TODO: find the strategy with the highest Sharpe Ratio (`best_sharpe`) and
#   the strategy with the lowest Max Drawdown (`best_mdd`) in `strat_only`.
#   > Hint: a pandas Series has methods for exactly this.
best_sharpe = strat_only.index[0]
best_mdd    = strat_only.index[0]

print(f'Best Sharpe  : {best_sharpe}  ({strat_only.loc[best_sharpe, "Sharpe Ratio"]:.3f})')
print(f'Lowest MDD   : {best_mdd}  ({strat_only.loc[best_mdd, "Max Drawdown (%)"]:.2f}%)')
print()
print(f'Transaction cost assumption  : {TRANSACTION_COST*100:.3f}% per trade (one-way)')
print(f'Risk-free rate               : 2% p.a. ({RISK_FREE_RATE*252*100:.4f}% daily)')
print(f'Test period                  : {df_test_eff["date"].min().date()} -> '
      f'{df_test_eff["date"].max().date()}  ({N_test_eff} trading days)')

print('\nReports saved to reports/:')
for fname in ['fig_cumulative_returns.png', 'fig_drawdown_profiles.png',
              'fig_signal_maps.png', 'fig_sharpe_mdd.png',
              'fig_monthly_heatmap.png', 'week5_strategy_performance.csv']:
    exists = os.path.exists(os.path.join(REPORTS_DIR, fname))
    print(f'  {"OK" if exists else "MISSING":6s}  {fname}')


---
## 10. Conclusions & Reflection

### Write-up

For each strategy, think:

1. What did your **initial** run look like (Sharpe, MDD, trade count) before you touched any
   hyperparameters? What did that tell you about what was wrong?
2. Which hyperparameter(s) mattered most once your logic was correct? How did you decide on your
   final values — trial and error, or reasoning about what the parameter controls?
3. For Strategy 3 specifically: how did you decide when a "normal-conviction" reading should be
   allowed to trade versus when you needed the extreme/regime-gated path? What would happen if you
   removed the regime gate entirely?
4. For Strategy 4 specifically: what did you observe about the interaction between the trailing
   stop and the entry logic when they were structured incorrectly? What behavior tipped you off?
5. Compare your best strategy's Sharpe Ratio and Max Drawdown to Buy & Hold. Is it actually better
   on a risk-adjusted basis, or just on total return? Which metric would you present to a risk
   manager, and why?

### Grading rubric
- Correctness: does each strategy implement the behavior described in its section, without
  altering function signatures or the backtesting/evaluation code?
- Tuning: is there evidence of deliberate, reasoned hyperparameter search (not just copy-pasting
  values)?
- Write-up: do your answers demonstrate understanding of *why* a change helped, not just *that*
  it helped?

### Week 6 Preview -- Reinforcement Learning Agent

These four rule-based strategies become the **baseline benchmarks** that a Reinforcement Learning
agent (PPO / DQN) must beat next week. The RL agent will:
- Observe the same feature vectors as our LSTM models.
- Learn position-sizing and entry/exit rules via reward signals (portfolio returns).
- Be rewarded for Sharpe improvement and penalised for excessive drawdowns.
- Compete head-to-head against all four strategies on the same test set.
